In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [20]:
target_df.OS_YEARS

ID
P132697    1.115068
P132698    4.928767
P116889    2.043836
P132699    2.476712
P132700    3.145205
             ...   
P121828         NaN
P121829         NaN
P121830    1.997260
P121853    0.095890
P121834    2.290411
Name: OS_YEARS, Length: 3323, dtype: float64

In [30]:
plt.figure()

sns.violinplot(x=target_df.OS_YEARS, y=df["MONOCYTES"])

plt.tight_layout()
plt.show()

KeyboardInterrupt: 

In [2]:
# Clinical Data
df = pd.read_csv("./X_train/clinical_train.csv", index_col=0)
df_eval = pd.read_csv("./X_test/clinical_test.csv", index_col=0)

# Molecular Data
maf_df = pd.read_csv("./X_train/molecular_train.csv", index_col=0)
maf_eval = pd.read_csv("./X_test/molecular_test.csv", index_col=0)

target_df = pd.read_csv("./target_train.csv", index_col=0)

In [3]:
df.head()

,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS
ID,,,,,,,,
P132697,MSK,14.0,2.8,0.2,0.7,7.6,119.0,"46,xy,del(20)(q12)[2]/46,xy[18]"
P132698,MSK,1.0,7.4,2.4,0.1,11.6,42.0,"46,xx"
P116889,MSK,15.0,3.7,2.1,0.1,14.2,81.0,"46,xy,t(3;3)(q25;q27)[8]/46,xy[12]"
P132699,MSK,1.0,3.9,1.9,0.1,8.9,77.0,"46,xy,del(3)(q26q27)[15]/46,xy[5]"
P132700,MSK,6.0,128.0,9.7,0.9,11.1,195.0,"46,xx,t(3;9)(p13;q22)[10]/46,xx[10]"


In [18]:
df.mean(axis=0)

TypeError: Cannot perform reduction 'mean' with string dtype

In [11]:
df.describe()

,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT
count,3214.000000,3051.000000,3130.000000,2722.000000,3213.000000,3199.000000
mean,5.982545,6.535164,3.264735,0.955868,9.893549,167.048900
std,7.615439,10.247219,5.237043,2.666478,2.041158,149.477031
min,0.000000,0.200000,0.000000,0.000000,4.000000,2.000000
25%,1.000000,2.700000,1.000000,0.150000,8.500000,65.500000
50%,3.000000,4.100000,2.000000,0.370000,9.700000,123.000000
75%,8.000000,6.655000,3.690000,0.783000,11.200000,229.500000
max,91.000000,154.400000,109.620000,44.200000,16.600000,1451.000000


In [5]:
# columns to drop in molecular data
cols_to_drop = [
    "CHR", # the length of the chromosome is not relevant to the survival rate
    "START", # the position of the mutation is not relevant to the survival rate
    "END",
    "REF", # does not provide any information about the mutation
    "ALT", 
    "DEPTH" # it is a measure of the perf of measurement from the lab, not relevant
]

test = maf_df.drop(columns=cols_to_drop, axis=0)

In [6]:
test

,GENE,PROTEIN_CHANGE,EFFECT,VAF
ID,,,,
P100000,CBL,p.C419Y,non_synonymous_codon,0.0830
P100000,IRF1,p.Y164*,stop_gained,0.0220
P100000,ROBO2,p.?,splice_site_variant,0.4100
P100000,TET2,p.R1262L,non_synonymous_codon,0.4300
P100000,DNMT3A,p.E505fs*141,frameshift_variant,0.0898
...,...,...,...,...
P131472,MLL,MLL_PTD,PTD,NaN
P131505,MLL,MLL_PTD,PTD,NaN
P131816,MLL,MLL_PTD,PTD,NaN


In [10]:
mol_agg = test.groupby("ID").agg(
    nb_mutations=("GENE", "count"),
    mean_vaf=("VAF", "mean"),
    max_vaf=("VAF", "max"),
)

mol_agg

,nb_mutations,mean_vaf,max_vaf
ID,,,
P100000,6,0.301300,0.7730
P100001,2,0.259500,0.4180
P100002,2,0.397000,0.5970
P100004,1,0.469100,0.4691
P100006,5,0.169300,0.3720
...,...,...,...
P132725,5,0.533600,0.8060
P132726,1,0.269000,0.2690
P132727,2,0.322250,0.3595


In [7]:
# On pivote pour avoir les gènes en colonnes et la VAF en valeur
# Si un patient a plusieurs fois le même gène (rare), on prend la VAF max
df_genes = test.pivot_table(index='ID', 
                             columns='GENE', 
                             values='VAF', 
                             aggfunc='max')

# On remplace les cases vides (NaN) par 0 (pas de mutation = VAF de 0)
df_genes = df_genes.fillna(0)
df_genes

GENE,ABL1,ARID1A,ARID2,ASXL1,ASXL2,ATRX,BAP1,BCL10,BCOR,BCORL1,...,TET2,TP53,U2AF1,U2AF2,WHSC1,WT1,ZBTB33,ZMYM3,ZNF318,ZRSR2
ID,,,,,,,,,,,,,,,,,,,,,
P100000,0.0,0.0,0.0,0.0000,0.000,0.0,0.0,0.0,0.0,0.0,...,0.4300,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P100001,0.0,0.0,0.0,0.0000,0.000,0.0,0.0,0.0,0.0,0.0,...,0.0000,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P100002,0.0,0.0,0.0,0.0000,0.000,0.0,0.0,0.0,0.0,0.0,...,0.0000,0.597,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P100004,0.0,0.0,0.0,0.0000,0.000,0.0,0.0,0.0,0.0,0.0,...,0.0000,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P100006,0.0,0.0,0.0,0.0000,0.000,0.0,0.0,0.0,0.0,0.0,...,0.0555,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
P132725,0.0,0.0,0.0,0.4510,0.000,0.0,0.0,0.0,0.0,0.0,...,0.0000,0.000,0.445,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P132726,0.0,0.0,0.0,0.0000,0.000,0.0,0.0,0.0,0.0,0.0,...,0.0000,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P132727,0.0,0.0,0.0,0.3595,0.000,0.0,0.0,0.0,0.0,0.0,...,0.0000,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
